# 33 - Temporal TDA: Tracking Topology Across Time

Real-world data is not static. Phase transitions, biological dynamics, financial markets — all produce topological spaces that *change* over time. **Temporal TDA** tracks these changes:
- When does a connected component merge? (H₀ bifurcation)
- When does a loop appear or die? (H₁ birth/death)
- When does the global topology undergo a *qualitative* change? (bifurcation event)

`pysurgery.core.temporal_topology` provides the full pipeline: from raw point cloud sequences to certified bifurcation events.

## Learning Goals
- Build zigzag complex sequences with `build_union_intersection_zigzag`
- Compute temporal homology with `compute_temporal_homology` → `TemporalBarcode`
- Detect bifurcation events with `detect_bifurcations` → `List[BifurcationEvent]`
- Run the full analysis pipeline with `analyze_temporal_evolution` → `TemporalAnalysisResult`
- Use `compute_topological_loss` to measure deviation from a target topology

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Point cloud time series $\{P_t\}_{t=0}^T$ | `List[np.ndarray]` |
| **Algebraic** | Parameter-indexed barcodes $\{B_k(t)\}$ | `TemporalBarcode` |
| **Result** | Bifurcation events with parameter values | `BifurcationEvent` |

## Formal Background

A **topological phase transition** at parameter $t^*$ is a discontinuity in the barcode diagram: the number of bars in $H_k$ changes abruptly. `detect_bifurcations` identifies these points by measuring the bottleneck distance between consecutive temporal barcodes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pysurgery.core.temporal_topology import (
    TemporalBarcode, build_union_intersection_zigzag,
    compute_temporal_homology,
    detect_bifurcations,
    analyze_temporal_evolution, TemporalAnalysisResult,
    compute_topological_loss,
)

print("=" * 70)
print("33 - Temporal TDA: Tracking Topology Across Time: Setup Complete")
print("=" * 70)

## Part 1: Building a Temporal Point Cloud

We simulate a topological phase transition: a single annular ring splitting into two rings as a parameter increases. This models phenomena like cell division, vortex splitting in fluid dynamics, or network cluster separation.

In [ ]:
rng = np.random.default_rng(12)

T = 12   # number of time steps
t_transition = 6   # phase transition occurs here
point_clouds = []

for t in range(T):
    if t < t_transition:
        # Single ring
        angles = np.linspace(0, 2 * np.pi, 30)
        pts = np.column_stack([
            1.5 * np.cos(angles),
            1.5 * np.sin(angles),
        ]) + rng.normal(0, 0.1, (30, 2))
    else:
        # Two rings, gradually separating
        sep = 1.0 + 0.3 * (t - t_transition)
        a = np.linspace(0, 2 * np.pi, 15)
        pts = np.vstack([
            np.column_stack([np.cos(a) - sep, np.sin(a)]) + rng.normal(0, 0.1, (15, 2)),
            np.column_stack([np.cos(a) + sep, np.sin(a)]) + rng.normal(0, 0.1, (15, 2)),
        ])
    point_clouds.append(pts)

print(f"Generated {T} time steps, phase transition at t={t_transition}")

# Visualise two representative frames
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, t, label in zip(axes, [t_transition - 1, t_transition + 1],
                              [f"t={t_transition-1} (1 ring)", f"t={t_transition+1} (2 rings)"]):
    ax.scatter(point_clouds[t][:, 0], point_clouds[t][:, 1],
               s=30, alpha=0.7, c="steelblue", edgecolors="k", lw=0.5)
    ax.set_title(label); ax.set_aspect("equal")
    ax.set_xlim(-4, 4); ax.set_ylim(-2.5, 2.5)
plt.suptitle("Temporal Point Cloud: Ring Splitting")
plt.tight_layout(); plt.show()

## Part 2: Zigzag Complex Sequence

The union-intersection zigzag
$$K_0 \hookrightarrow K_0 \cup K_1 \hookleftarrow K_1 \hookrightarrow K_1 \cup K_2 \hookleftarrow \cdots$$
captures both the current topology (union) and the transitions (intersection). It is the canonical input to zigzag persistence.

In [ ]:
# Build zigzag sequence
complex_seq = build_union_intersection_zigzag(
    point_clouds,
    epsilon=0.6,          # Rips threshold
    max_dimension=2,
    complex_type="Rips",
)
print(f"Zigzag sequence: {len(complex_seq)} complexes")
for i in range(0, len(complex_seq), 3):
    K = complex_seq[i]
    print(f"  K_{i}: {K.num_simplices(0)} vertices, {K.num_simplices(1)} edges, {K.num_simplices(2)} triangles")

## Part 3: Temporal Homology

`compute_temporal_homology` runs zigzag persistence on the complex sequence and returns a `TemporalBarcode` — a list of barcodes indexed by parameter values. Each barcode gives the H_k diagram at that time step.

In [ ]:
parameters = list(range(T))

# H₀: connected components
tb0: TemporalBarcode = compute_temporal_homology(
    complex_seq, dimension=0, parameters=parameters, field="Z2"
)
# H₁: loops
tb1: TemporalBarcode = compute_temporal_homology(
    complex_seq, dimension=1, parameters=parameters, field="Z2"
)

print(f"H₀ temporal barcodes ({len(tb0.barcodes)} bars):")
for b in tb0.barcodes:
    print(f"  [{b.birth:.0f}, {b.death:.0f})  ← connected component")

print()
print(f"H₁ temporal barcodes ({len(tb1.barcodes)} bars):")
for b in tb1.barcodes:
    death_str = f"{b.death:.0f}" if b.death < 999 else "∞"
    print(f"  [{b.birth:.0f}, {death_str})  ← loop")

print()
print("Expected: 1 long H₁ bar for t<6, then 2 H₁ bars for t≥6 (ring splitting)")

## Part 4: Bifurcation Detection

`detect_bifurcations` compares consecutive temporal barcodes by bottleneck distance. A sudden jump above a threshold indicates a topological phase transition. The `BifurcationEvent` records the parameter value, dimension, and event type (birth/death/merge/split).

In [ ]:
all_barcodes = [tb0, tb1]
bifurcations = detect_bifurcations(all_barcodes, threshold=0.5)

print(f"Detected {len(bifurcations)} bifurcation events:")
for b in bifurcations:
    print(f"  t={b.parameter:.1f}  dim=H_{b.dimension}  type={b.event_type}")
    print(f"  description: {b.description}")
print()
print(f"Expected: bifurcation at t≈{t_transition} in H₁ (1 loop → 2 loops)")

# Visualise the bottleneck distances over time
fig, ax = plt.subplots(figsize=(9, 3))
for dim, tb, color in [(0, tb0, "steelblue"), (1, tb1, "darkorange")]:
    t_vals = list(range(len(tb.barcodes) - 1))
    dists = [tb.bottleneck_distance(tb.barcodes[i], tb.barcodes[i+1])
             for i in t_vals] if hasattr(tb, "bottleneck_distance") else [0] * len(t_vals)
    ax.plot(t_vals, dists, "o-", color=color, lw=2, label=f"H_{dim}")
ax.axvline(t_transition - 0.5, color="crimson", ls="--", lw=2, label="Phase transition")
ax.set_xlabel("Time step"); ax.set_ylabel("Bottleneck distance")
ax.set_title("Topological Phase Transition Detection")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Part 5: Full Temporal Analysis

`analyze_temporal_evolution` runs the full pipeline: zigzag complex construction → temporal homology for all dimensions → bifurcation detection → summary report. This is the recommended entry point for exploratory analysis.

In [ ]:
result: TemporalAnalysisResult = analyze_temporal_evolution(
    point_clouds,
    epsilon=0.6,
    max_dimension=2,
    parameters=parameters,
    field="Z2",
)

print("TemporalAnalysisResult:")
print(f"  phase_transitions:  {result.phase_transitions}")
print(f"  persistent_features: {result.persistent_features}")
print(f"  dominant_dimension:  {result.dominant_dimension}")
print(f"  exact:               {result.exact}")
print(f"  theorem_tag:         {result.theorem_tag}")
print()

# Topological loss against a target (1 loop throughout)
from pysurgery.core.persistent_homology import Barcode
target = [Barcode(birth=0.0, death=float(T))]
loss = compute_topological_loss(tb1, target, epsilon=0.1)
print(f"Topological loss (target: 1 persistent loop) = {loss:.4f}")
print("High loss confirms the deviation from 1-loop topology at the phase transition.")

## Summary Checklist

- [x] Generated temporal point cloud sequences and visualised a phase transition
- [x] Built zigzag complex sequences with `build_union_intersection_zigzag`
- [x] Computed temporal homology in degrees 0 and 1
- [x] Detected bifurcation events with `detect_bifurcations`
- [x] Read `BifurcationEvent.parameter`, `.dimension`, `.event_type`
- [x] Used `analyze_temporal_evolution` for the full one-call pipeline
- [x] Used `compute_topological_loss` to measure topological deviation

## Next Steps
- **NB 27**: Persistent homology refined — deeper coverage of exact barcodes and zigzag
- **NB 34**: The capstone uses temporal analysis for time-varying input data